**Forecasting Results Analysis – Quick Guide**

---

### Purpose

This notebook aggregates and analyzes the results from the parallel `papermill` experiments to produce final, paper-ready tables.

---

### Workflow

1.  **Load & Combine (`load_all_results`)**
    *   Scans the `papermill_outputs/results/` directory for all `results-*.csv` files.
    *   Loads and concatenates all individual results into a single master pandas DataFrame.

2.  **Find Best Configurations (`find_best_configs`)**
    *   **Groups data by:** `(Method, Configuration)`.
    *   **Calculates robustness:** Aggregates by finding the mean of the chosen `BEST_METRIC` across all wells for each group.
    *   **Identifies winner:** For each `Method`, it selects the `Configuration` with the best (lowest) mean score. This produces the summary table of winning configurations.

3.  **Generate Final Report (`generate_final_report`)**
    *   Uses the list of winning configurations from the previous step as a filter.
    *   Selects all rows from the complete results DataFrame that match a winning `(Method, Configuration)` pair.
    *   Formats this selection into the final, clean `Well | Method | R² | SMAPE | MAE` table for publication.

In [ ]:
import pandas as pd
from pathlib import Path
import glob
import numpy as np
from evaluation.evaluation  import display_metrics

# --- Configuration ---
# This must match the directory used by the orchestrator, results_56 or results_112
RESULTS_DIR = Path("papermill_outputs/results_56/")

# Define the primary metric for selecting the "best" model.
# Use the column name from your CSV files (e.g., 'SMAPE', 'MAE').
# Lower is better for these metrics.
BEST_METRIC = 'SMAPE'

def load_all_results(results_dir: Path) -> pd.DataFrame:
    """
    Scans a directory for result CSVs, loads them, and combines them
    into a single DataFrame.
    """
    all_csv_files = list(results_dir.glob("results-*.csv"))
    
    if not all_csv_files:
        print(f"Error: No result files found in '{results_dir}'.")
        print("Please make sure the orchestrator has run successfully.")
        return pd.DataFrame()

    df_list = [pd.read_csv(f) for f in all_csv_files]
    full_df = pd.concat(df_list, ignore_index=True)

    print(f"Successfully loaded and combined {len(df_list)} result files.")
    print(f"Total results found: {len(full_df)}")
    
    return full_df

def preprocess_results(df: pd.DataFrame) -> pd.DataFrame:
    """
    Performs initial cleaning and preparation of the raw results DataFrame.
    """
    # Ensure the SMAPE column is numeric (it might be a string like '16.92%')
    if df['SMAPE'].dtype == 'object':
        df['SMAPE'] = df['SMAPE'].str.replace('%', '').astype(float)
        
    # We only care about the cumulative metrics for the final table
    analysis_df = df[df['Category'] == 'Cumulative'].copy()
    
    # Rename columns for clarity if needed
    analysis_df = analysis_df.rename(columns={'Well': 'Well', 'Method': 'Method', 'Setting': 'Configuration'})
    
    return analysis_df

# --- Load and Preprocess Data ---
raw_df = load_all_results(RESULTS_DIR)
if not raw_df.empty:
    analysis_df = preprocess_results(raw_df)
    print("\nSample of preprocessed data:")
    display(analysis_df.head())

In [ ]:
def find_best_configs(df: pd.DataFrame, metric: str) -> pd.DataFrame:
    """
    Analyzes the results to find the best configuration for each model.
    
    Args:
        df (pd.DataFrame): The preprocessed DataFrame of results.
        metric (str): The name of the metric column to use for optimization (lower is better).

    Returns:
        pd.DataFrame: A DataFrame containing the best configuration for each model.
    """
    # 1. Calculate average performance across all wells for each model/config pair
    metric_mean = f'Mean_{metric}'
    metric_std = f'Std_{metric}'
    
    summary_table = df.groupby(['Method', 'Configuration']).agg(
        **{metric_mean: (metric, 'mean'),
           metric_std: (metric, 'std')}
    ).reset_index()

    print("--- Summary: Average Performance Across All Wells ---")
    display(summary_table.sort_values(by=['Method', metric_mean]))

    # 2. Find the index of the best configuration for each model
    best_indices = summary_table.groupby('Method')[metric_mean].idxmin()
    best_configs_df = summary_table.loc[best_indices].copy()
    
    print(f"\n\n--- Final Table: Best Configuration per Model (Lowest Avg. {metric}) ---")
    display(best_configs_df[['Method', 'Configuration', metric_mean, metric_std]])
    
    return best_configs_df

# --- Run Analysis ---
if 'analysis_df' in locals() and not analysis_df.empty:
    best_configs = find_best_configs(analysis_df, BEST_METRIC)

In [ ]:
def generate_final_report(full_results_df: pd.DataFrame, best_configs_df: pd.DataFrame) -> pd.DataFrame:
    """
    Generates the final report table containing only the results from the
    best configuration for each model.

    Args:
        full_results_df (pd.DataFrame): The complete, preprocessed results DataFrame.
        best_configs_df (pd.DataFrame): The DataFrame identifying the best config for each model.

    Returns:
        pd.DataFrame: The final, clean report table.
    """
    # Create a 'key' in both dataframes to perform a merge (like a SQL join)
    full_results_df['join_key'] = full_results_df['Method'] + full_results_df['Configuration']
    best_configs_df['join_key'] = best_configs_df['Method'] + best_configs_df['Configuration']

    # Keep only the rows from the full results that match a "best config" key
    final_df = full_results_df[full_results_df['join_key'].isin(best_configs_df['join_key'])].copy()

    # Select and reorder columns to match the desired final format
    report_columns = ['Well', 'Method', 'R²', 'SMAPE', 'MAE']
    final_report = final_df[report_columns]

    # Sort for consistent presentation
    final_report = final_report.sort_values(by=['Well', 'Method']).reset_index(drop=True)

    return final_report

# --- Generate and Display Final Report ---
if 'best_configs' in locals() and not best_configs.empty:
    print("\n\n--- Building Final Report from Best Configurations ---")
    final_paper_table = generate_final_report(analysis_df, best_configs)
    
    print("\n--- Final Report Table for Paper ---")
display_metrics(final_paper_table)